# SE446 -- Big Data Engineering | Week 11 Lab
## Spark Structured Streaming with Kafka

**Course:** SE446 Big Data Engineering -- Alfaisal University  
**Instructor:** Prof. Anis Koubaa  
**Dataset:** Chicago Crimes (used throughout the course)  
**Prerequisite:** Week 10 Lab A (Kafka Fundamentals)

---

### What you will learn in this lab

| Part | Concept | Why it matters |
|------|---------|---------------|
| 1 | Producing crime events | Setting up the live data source for streaming |
| 2 | Reading from Kafka with `readStream` | How Spark ingests a continuous data source |
| 3 | Parsing JSON messages | Schema definition and deserialization |
| 4 | Streaming aggregations | Stateful processing -- counts and sums on live data |
| 5 | Time windows | Tumbling windows for time-based grouping |
| 6 | Watermarks | Handling late data and managing memory |
| 7 | Output modes | Complete vs Update vs Append |
| 8 | Checkpointing | Fault tolerance and crash recovery |

> **Important:** Run cells in order. Practice cells are wrapped in `try/except` and will never block execution.  
> All streaming queries use `trigger(once=True)` so they execute like batch queries in the notebook -- no infinite blocking.

---
## Part 0: Setup

We need three libraries:
- `pyspark` -- the Python API for Apache Spark (includes Structured Streaming)
- `confluent-kafka` -- to produce test events into Kafka (same as Week 10)
- `paramiko` -- to create a secure SSH tunnel to the cluster

In [ ]:
!pip install pyspark confluent-kafka paramiko -q

### Connect to the Cluster

The cell below creates a **secure SSH tunnel** to the Kafka broker -- identical to the Week 10 lab.

- You will be prompted for the **cluster IP**, your **cluster username**, and **password** (all provided by your instructor).
- Alternatively, set the `CLUSTER_IP` environment variable before running the notebook to skip the IP prompt.
- The tunnel maps `localhost:9092` on your machine to the Kafka broker on the cluster.
- PySpark will connect to Kafka through this tunnel.

**Replace `YOUR_NAME`** with your actual name or student ID.

In [ ]:
STUDENT_ID = "YOUR_NAME"   # <-- CHANGE THIS to your name or student ID

# ============================================================
# SSH Tunnel Setup (secure connection to the Kafka broker)
# ============================================================
import os, json, uuid, time, getpass, threading, random
from datetime import datetime, timedelta
import paramiko

# The cluster IP is provided by the instructor (do not hardcode in the notebook).
# Options to supply it:
#   1. Set environment variable CLUSTER_IP before running the notebook, OR
#   2. Enter it at the prompt below.
MASTER_IP  = os.environ.get("CLUSTER_IP", "").strip()
LOCAL_PORT = 9092
REMOTE_PORT = 9092

# Check if tunnel is already open
import socket
def _port_open(port):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)
        s.connect(("localhost", port))
        s.close()
        return True
    except:
        return False

if _port_open(LOCAL_PORT):
    print(f"Port {LOCAL_PORT} already open -- reusing existing tunnel.")
else:
    print("=== SSH Tunnel Setup ===")
    if not MASTER_IP:
        MASTER_IP = getpass.getpass("Cluster IP (provided by instructor): ").strip()
    ssh_user = input("SSH Username: ")
    ssh_pass = getpass.getpass("SSH Password: ")

    # Create SSH client
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(MASTER_IP, port=22, username=ssh_user, password=ssh_pass)

    # Open port-forward in a background thread
    transport = ssh.get_transport()
    class _TunnelServer(threading.Thread):
        daemon = True
        def run(self):
            server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            server.bind(("localhost", LOCAL_PORT))
            server.listen(5)
            while True:
                client, addr = server.accept()
                chan = transport.open_channel("direct-tcpip",
                    ("localhost", REMOTE_PORT), addr)
                threading.Thread(target=_forward, args=(client, chan), daemon=True).start()

    def _forward(local, remote):
        import select
        while True:
            r, _, _ = select.select([local, remote], [], [], 1)
            if local in r:
                data = local.recv(4096)
                if not data: break
                remote.sendall(data)
            if remote in r:
                data = remote.recv(4096)
                if not data: break
                local.sendall(data)
        local.close()
        remote.close()

    _TunnelServer().start()
    time.sleep(1)
    print(f"Tunnel open: localhost:{LOCAL_PORT} -> cluster:{REMOTE_PORT}")

BROKER = f"localhost:{LOCAL_PORT}"
TOPIC  = f"streaming-{STUDENT_ID}"

print(f"\nBroker:     {BROKER}")
print(f"Topic:      {TOPIC}")
print(f"Student ID: {STUDENT_ID}")

### Create SparkSession

A **SparkSession** is the single entry point for all Spark functionality -- batch and streaming alike. We configure it with the Kafka connector package so Spark knows how to read from and write to Kafka.

The package `spark-sql-kafka-0-10_2.12:3.5.4` provides:
- `readStream.format("kafka")` -- to read streaming data from Kafka
- `writeStream.format("kafka")` -- to write streaming results back to Kafka

Without this package, Spark would not know how to speak the Kafka protocol.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName(f"SE446-Streaming-{STUDENT_ID}") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"App name:      {spark.sparkContext.appName}")
print(f"Master:        {spark.sparkContext.master}")
print("\nSparkSession is ready.")

---
## Part 1: Producing Crime Events (Concept: Setting Up the Data Source)

### Why do we need a producer in a streaming lab?

Spark Structured Streaming **reads** from Kafka -- it is a consumer. But before we can read, we need data in the topic. In Week 10, we learned how to produce messages with keys and observe partition assignment. Here, we use the same technique to create a **realistic crime event stream** that Spark will process.

Each event has:
- `id` -- unique crime identifier
- `type` -- crime category (THEFT, BATTERY, ASSAULT, etc.)
- `district` -- police district number (used as the Kafka key for partition affinity)
- `hour` -- hour of day (0--23)
- `arrest` -- whether an arrest was made
- `event_time` -- when the crime occurred (used for time-windowed aggregations)

We produce **30 events** with timestamps spread over a 30-minute window so we have enough data for meaningful time-based analysis in later parts.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer

# --- Create topic with 3 partitions ---
admin = AdminClient({"bootstrap.servers": BROKER})
new_topic = NewTopic(TOPIC, num_partitions=3, replication_factor=1)
futures = admin.create_topics([new_topic])

for topic_name, future in futures.items():
    try:
        future.result()
        print(f"Topic '{topic_name}' created successfully!")
    except Exception as e:
        print(f"Topic '{topic_name}': {e}")

# --- Produce 30 crime events ---
producer = Producer({"bootstrap.servers": BROKER})

CRIME_TYPES = ["THEFT", "BATTERY", "ASSAULT", "ROBBERY", "BURGLARY",
               "NARCOTICS", "CRIMINAL DAMAGE", "MOTOR VEHICLE THEFT"]
DISTRICTS   = [1, 2, 3, 5, 7, 8, 11, 12]

base_time = datetime(2026, 4, 18, 14, 0, 0)  # Start at 14:00

delivery_count = {"success": 0, "fail": 0}

def delivery_report(err, msg):
    if err:
        delivery_count["fail"] += 1
    else:
        delivery_count["success"] += 1

print(f"Producing 30 crime events to topic '{TOPIC}'...\n")

events = []
for i in range(30):
    event_time = base_time + timedelta(minutes=random.randint(0, 30))
    event = {
        "id": i + 1,
        "type": random.choice(CRIME_TYPES),
        "district": random.choice(DISTRICTS),
        "hour": event_time.hour,
        "arrest": random.choice([True, False]),
        "event_time": event_time.strftime("%Y-%m-%d %H:%M:%S")
    }
    events.append(event)
    key = str(event["district"])
    value = json.dumps(event)
    producer.produce(TOPIC, key=key, value=value, callback=delivery_report)

remaining = producer.flush(timeout=15)

print(f"Delivered: {delivery_count['success']}")
print(f"Failed:    {delivery_count['fail']}")
print(f"Remaining: {remaining}")

# Show a few sample events
print(f"\n--- Sample Events ---")
for e in events[:5]:
    print(f"  id={e['id']:>2}, type={e['type']:<20s} district={e['district']:>2}, "
          f"hour={e['hour']:>2}, arrest={str(e['arrest']):<5s}, time={e['event_time']}")

### Comprehension Check

**Q1: Why do we use the district number as the Kafka message key? What guarantee does this give us?**

> *Hint: Think about hash partitioning from Week 10 -- all events from the same district go to the same partition, preserving per-district ordering.*

**Q2: We spread event_time over a 30-minute window. Why is this important for the time-windowed aggregations we will do in Parts 5 and 6?**

> *If all events had the exact same timestamp, every event would fall into one window -- not very useful for demonstrating windowing.*

---
## Part 2: Reading from Kafka with readStream (Concept: Streaming DataFrames)

### Batch vs Streaming: One Line Changes Everything

In batch Spark, you read data with `spark.read`. In streaming Spark, you read with `spark.readStream`. That is the **only difference** in the read API:

```python
# Batch
df = spark.read.format("kafka").option(...).load()

# Streaming
df = spark.readStream.format("kafka").option(...).load()
```

The streaming DataFrame is conceptually an **unbounded table** that grows as new messages arrive in Kafka. Spark processes new rows incrementally -- it does not re-read the entire topic every time.

### The Raw Kafka Schema

When Spark reads from Kafka, it produces a DataFrame with a **fixed schema** -- regardless of what your messages contain:

| Column | Type | Description |
|--------|------|-------------|
| key | binary | The message key (bytes) |
| value | binary | The message value (bytes) |
| topic | string | Which topic this came from |
| partition | integer | Which partition |
| offset | long | The offset within the partition |
| timestamp | timestamp | Kafka-assigned timestamp |
| timestampType | integer | 0=create time, 1=log append time |

Notice that **key and value are binary** -- raw bytes. Kafka does not know or care about your message format. You must cast and parse them yourself (we do this in Part 3).

In [ ]:
# Read from Kafka as a streaming DataFrame
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()

# Print the schema -- notice key and value are BINARY
print("=== Raw Kafka Stream Schema ===")
raw_stream.printSchema()

print(f"Is streaming: {raw_stream.isStreaming}")

In [ ]:
# Write to memory table with trigger(once=True) so we can inspect the raw data
query = raw_stream \
    .selectExpr("CAST(key AS STRING) AS key",
                "CAST(value AS STRING) AS value",
                "topic", "partition", "offset", "timestamp") \
    .writeStream \
    .format("memory") \
    .queryName("raw_kafka") \
    .trigger(once=True) \
    .start()

query.awaitTermination()

print("=== Raw Messages from Kafka ===")
spark.sql("SELECT key, value, partition, offset, timestamp FROM raw_kafka ORDER BY partition, offset").show(10, truncate=60)

### Comprehension Check

**Q1: Why are the key and value columns binary (bytes) instead of string or JSON? What does this tell you about Kafka's design philosophy?**

> *Hint: Kafka is format-agnostic. It stores and transmits raw bytes. The producer and consumer agree on the serialization format (JSON, Avro, Protobuf, etc.) -- Kafka itself does not interpret the data.*

**Q2: We used `startingOffsets = "earliest"`. What would happen if we used `"latest"` instead? When would you choose each option?**

> *Think about: do you want historical data (earliest) or only future data (latest)?*

---
## Part 3: Parsing JSON Messages (Concept: Schema Definition + Deserialization)

### From Raw Bytes to Structured Data

The raw Kafka value is a JSON string encoded as bytes. To work with it as structured data, we need to:

1. **Cast** the binary value to a string: `CAST(value AS STRING)`
2. **Define a schema** that matches our JSON structure (using `StructType`)
3. **Parse** the JSON string into columns using `from_json()`
4. **Flatten** the nested struct into individual columns with `select("data.*")`

This is exactly the same concept as `json.loads()` in the Python consumer from Week 10 -- but done declaratively at scale.

```python
# Week 10 (Python consumer)
event = json.loads(msg.value())
crime_type = event["type"]

# Week 11 (Spark Structured Streaming)
parsed = df.select(from_json(col("value"), schema).alias("data")).select("data.*")
# Now you have a column called "type" -- same data, just at scale
```

### Why define the schema manually?

Spark **cannot** infer the schema of a streaming source (unlike batch, where it can peek at the data). You must tell Spark exactly what fields to expect and their types. If the schema is wrong, the columns will be null.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, TimestampType
from pyspark.sql.functions import from_json, col

# Define the schema matching our crime event JSON
crime_schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("type",       StringType(),  True),
    StructField("district",   IntegerType(), True),
    StructField("hour",       IntegerType(), True),
    StructField("arrest",     BooleanType(), True),
    StructField("event_time", StringType(),  True),  # We parse to timestamp below
])

# Read again from Kafka and parse
parsed_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(key AS STRING) AS district_key",
                "CAST(value AS STRING) AS json_value",
                "partition", "offset") \
    .select(
        col("district_key"),
        from_json(col("json_value"), crime_schema).alias("data"),
        col("partition"),
        col("offset")
    ) \
    .select(
        "data.id", "data.type", "data.district", "data.hour",
        "data.arrest",
        col("data.event_time").cast("timestamp").alias("event_time"),
        "partition", "offset"
    )

# Write to memory and display
query = parsed_stream.writeStream \
    .format("memory") \
    .queryName("parsed_crimes") \
    .trigger(once=True) \
    .start()

query.awaitTermination()

print("=== Parsed Crime Events ===")
spark.sql("SELECT * FROM parsed_crimes ORDER BY id").show(15, truncate=False)

print("\n=== Parsed Schema ===")
spark.sql("SELECT * FROM parsed_crimes").printSchema()

### What just happened?

We transformed the raw Kafka binary data into a structured DataFrame with typed columns:

```
Binary bytes  -->  JSON string  -->  Struct  -->  Individual columns
(value)           (CAST)           (from_json)    (select data.*)
```

Notice that `event_time` is now a proper **timestamp** column, not a string. This is critical for Part 5 (time windows) -- Spark needs a real timestamp to group events by time.

### Practice: Add a Derived Column

Parse the stream again, but add a new column called `time_of_day` that categorizes the hour:
- hour 6--11 = "MORNING"
- hour 12--17 = "AFTERNOON"
- hour 18--23 = "EVENING"
- hour 0--5 = "NIGHT"

Use `pyspark.sql.functions.when()` for the conditional logic.

In [ ]:
# PRACTICE: Parse the stream and add a time_of_day column
# Hint: Use .withColumn("time_of_day", when(col("hour") < 6, "NIGHT").when(...).otherwise(...))
try:
    from pyspark.sql.functions import when
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Practice skipped or error: {e}")

### Comprehension Check

**Q1: Why must we define the schema manually with `StructType` instead of letting Spark infer it automatically?**

> *Hint: In streaming, data arrives continuously. Spark cannot scan ahead to infer the schema -- it needs to know the structure before the first message arrives.*

**Q2: What would happen if we defined `event_time` as `IntegerType()` in the schema instead of `StringType()`? Would Spark crash or silently produce nulls?**

> *Spark uses `from_json` with a permissive mode by default -- mismatched types produce nulls, not errors. This is dangerous because you might not notice the problem.*

---
## Part 4: Streaming Aggregations (Concept: Stateful Processing)

### The Power of Structured Streaming: Same Code, Different Execution

This is the most important idea in Spark Structured Streaming:

> **You write the same DataFrame/SQL code as batch. Spark handles the incremental execution automatically.**

A `groupBy("type").count()` in batch scans the entire dataset once. In streaming, Spark:
1. Processes only the **new** messages since the last trigger.
2. **Updates** the running counts in internal state.
3. Outputs the results according to the output mode.

This is **stateful processing** -- Spark maintains state (the counts) across micro-batches.

### Output Mode: Complete

We use `outputMode("complete")` which means: output the **entire result table** after each trigger. For aggregations, this gives you the full, up-to-date counts every time.

In [ ]:
from pyspark.sql.functions import count, sum as spark_sum

# Re-read and parse the stream
crime_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp"))

# --- Aggregation 1: Crime count by type ---
crimes_by_type = crime_stream.groupBy("type").count()

query1 = crimes_by_type.writeStream \
    .format("memory") \
    .queryName("crimes_by_type") \
    .outputMode("complete") \
    .trigger(once=True) \
    .start()

query1.awaitTermination()

print("=== Crime Count by Type ===")
spark.sql("SELECT type, count FROM crimes_by_type ORDER BY count DESC").show(truncate=False)

In [ ]:
# --- Aggregation 2: District stats (count + arrest count) ---
district_stats = crime_stream.groupBy("district").agg(
    count("*").alias("total_crimes"),
    spark_sum(col("arrest").cast("int")).alias("total_arrests")
)

query2 = district_stats.writeStream \
    .format("memory") \
    .queryName("district_stats") \
    .outputMode("complete") \
    .trigger(once=True) \
    .start()

query2.awaitTermination()

print("=== District Crime Stats ===")
spark.sql("""
    SELECT district, total_crimes, total_arrests,
           ROUND(total_arrests * 100.0 / total_crimes, 1) AS arrest_rate_pct
    FROM district_stats
    ORDER BY total_crimes DESC
""").show(truncate=False)

### Observe the Results

Notice that the code is **identical** to what you would write in batch Spark:

```python
# Batch version
df.groupBy("type").count().show()

# Streaming version -- same!
stream.groupBy("type").count()
```

The only difference is how you **output** the results: `writeStream` instead of `show()`. Spark internally manages the state -- it remembers the running counts and updates them as new messages arrive.

### Comprehension Check

**Q1: We used `outputMode("complete")`. Why can't we use `outputMode("append")` for an aggregation query like `groupBy().count()`?**

> *Hint: In append mode, rows are written once and never updated. But a running count changes every time a new message arrives for that group. "Append" would mean writing the count of 5, then 6, then 7 as separate rows -- that is not what we want.*

**Q2: If 10 new THEFT events arrive in the next micro-batch, what happens to the THEFT row in the `crimes_by_type` table?**

> *The count is updated in Spark's internal state. In complete mode, the entire table (with the updated count) is re-emitted.*

---
## Part 5: Time Windows (Concept: Tumbling Windows)

### Event Time vs Processing Time

There are two notions of time in stream processing:

- **Event time** -- when the event actually happened (the `event_time` field in our crime data). This is embedded in the data itself.
- **Processing time** -- when Spark processes the event. This depends on network delays, batch scheduling, etc.

For analytics, **event time is almost always what you want**. A crime that happened at 14:05 should be counted in the 14:00--14:10 window, regardless of whether Spark processes it at 14:05 or 14:30.

### Tumbling Windows

A **tumbling window** divides the time axis into fixed-size, non-overlapping intervals:

```
Time:    14:00    14:05    14:10    14:15    14:20
         |--------|--------|--------|--------|
         Window 1  Window 2  Window 3  Window 4
         (5 min)   (5 min)   (5 min)   (5 min)
```

Each event falls into exactly one window based on its `event_time`. Spark uses the `window()` function to create these windows automatically.

In [ ]:
from pyspark.sql.functions import window

# Re-read and parse the stream
crime_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp"))

# --- Tumbling window: 5-minute windows grouped by crime type ---
windowed_counts = crime_stream \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("type")
    ) \
    .count()

query = windowed_counts.writeStream \
    .format("memory") \
    .queryName("windowed_crimes") \
    .outputMode("complete") \
    .trigger(once=True) \
    .start()

query.awaitTermination()

print("=== Crime Counts in 5-Minute Tumbling Windows ===")
spark.sql("""
    SELECT window.start AS window_start,
           window.end   AS window_end,
           type,
           count
    FROM windowed_crimes
    ORDER BY window_start, count DESC
""").show(20, truncate=False)

### Understanding the Output

Each row shows a **(window, crime_type)** pair with a count. Notice:

- The `window_start` and `window_end` columns define the 5-minute interval.
- Events are assigned to windows based on `event_time`, NOT when Spark processed them.
- If two THEFT events occurred at 14:02 and 14:04, they both fall in the [14:00, 14:05) window.

This is a **tumbling window** -- no overlap, no gaps. Every event belongs to exactly one window.

### Practice: Try a 1-Minute Window

Change the window size from 5 minutes to 1 minute. How does this affect the number of windows and the counts per window?

In [ ]:
# PRACTICE: Run the same windowed aggregation with a 1-minute window
# How many more windows do you get? Are the counts smaller per window?
try:
    # YOUR CODE HERE
    # Hint: Just change "5 minutes" to "1 minute" in the window() call
    pass
except Exception as e:
    print(f"Practice skipped or error: {e}")

### Comprehension Check

**Q1: What is the difference between a tumbling window and a sliding window?**

> *A tumbling window has no overlap -- each event belongs to exactly one window. A sliding window overlaps -- an event can belong to multiple windows. Example: a 10-minute window sliding every 2 minutes means 5 overlapping windows cover any point in time.*

**Q2: Why do we use `event_time` (from the data) instead of Spark's processing time for the window? Give a real-world scenario where using processing time would give wrong results.**

> *Imagine a crime occurs at 14:05 but the report arrives at the server at 14:30 due to a network delay. Using processing time, the crime would be counted in the 14:25--14:30 window -- incorrect. Using event time, it lands in the 14:05--14:10 window -- correct.*

---
## Part 6: Watermarks (Concept: Late Data + Memory Management)

### The Problem: Unbounded State

In Part 5, Spark kept track of **every** window it had ever seen. As time goes on, the number of windows grows forever. Spark must keep them all in memory because a late-arriving event could update any past window.

This is a memory problem. In a production system running for months, the state would grow until the application crashes.

### The Solution: Watermarks

A **watermark** tells Spark: "I promise that events will not arrive more than X minutes late. You can safely discard state for windows that are older than X minutes behind the latest event time."

```python
.withWatermark("event_time", "10 minutes")
```

This means:
- If the latest event time Spark has seen is 14:30, the watermark is at 14:20.
- Any event with `event_time < 14:20` is considered **late** and is dropped.
- Spark can discard all window state before 14:20.

### Trade-off

- **Smaller watermark** (e.g., 1 minute) = less memory, but more late events dropped.
- **Larger watermark** (e.g., 1 hour) = more late events accepted, but more memory used.

In practice, you choose the watermark based on your data's lateness profile. For crime data reported via mobile, a 10-minute watermark is reasonable.

In [ ]:
# Re-read and parse the stream
crime_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp"))

# --- Watermarked windowed aggregation ---
watermarked_counts = crime_stream \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("type")
    ) \
    .count()

query = watermarked_counts.writeStream \
    .format("memory") \
    .queryName("watermarked_crimes") \
    .outputMode("complete") \
    .trigger(once=True) \
    .start()

query.awaitTermination()

print("=== Watermarked Windowed Counts ===")
print("(Same results as Part 5 because our data has no late events)\n")
spark.sql("""
    SELECT window.start AS window_start,
           window.end   AS window_end,
           type,
           count
    FROM watermarked_crimes
    ORDER BY window_start, count DESC
""").show(20, truncate=False)

### Observe the Results

The results are **identical** to Part 5 because our test data does not contain any late events -- all events arrive in order. The watermark only changes behavior when events arrive out of order or late.

In production with real crime data:
- A crime reported 15 minutes after it happened would be dropped (watermark is 10 minutes).
- A crime reported 5 minutes late would be accepted.

### Comprehension Check

**Q1: What happens to an event whose `event_time` is older than the current watermark? Is it silently dropped or does Spark throw an error?**

> *It is silently dropped. Spark does not throw an error -- the event simply does not update any state. This is by design: the watermark is a contract, and the application accepts the trade-off.*

**Q2: Without a watermark, what would happen to Spark's memory usage as the streaming job runs for days?**

> *Memory grows indefinitely because Spark must keep state for every window ever created -- a window from 3 days ago might still get a late event. Eventually, the driver or executor runs out of memory and crashes.*

---
## Part 7: Output Modes Comparison (Concept: Complete vs Update vs Append)

### Three Ways to Emit Results

Spark Structured Streaming has three output modes:

| Mode | Behavior | Best for |
|------|----------|----------|
| **Complete** | Output the **entire** result table after each trigger | Dashboards that show full state |
| **Update** | Output only the **changed rows** since the last trigger | Alerts, notifications |
| **Append** | Output only **new rows** that will never change again | Writing to files, data lakes |

**Key constraint:** Aggregation queries **cannot** use append mode (because counts keep changing). Append mode works for non-aggregation queries (filters, maps) or for watermarked windows where finalized windows are appended.

Let us see the difference between complete and update.

In [ ]:
# Re-read and parse the stream
crime_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp"))

# --- Complete Mode: Output the FULL result table ---
type_counts = crime_stream.groupBy("type").count()

query_complete = type_counts.writeStream \
    .format("memory") \
    .queryName("mode_complete") \
    .outputMode("complete") \
    .trigger(once=True) \
    .start()

query_complete.awaitTermination()

print("=== COMPLETE Mode: Full Result Table ===")
print("(Every type is shown, even if it was not updated in this batch)\n")
spark.sql("SELECT type, count FROM mode_complete ORDER BY count DESC").show(truncate=False)

In [ ]:
# --- Update Mode: Output only CHANGED rows ---
crime_stream2 = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp"))

type_counts2 = crime_stream2.groupBy("type").count()

query_update = type_counts2.writeStream \
    .format("memory") \
    .queryName("mode_update") \
    .outputMode("update") \
    .trigger(once=True) \
    .start()

query_update.awaitTermination()

print("=== UPDATE Mode: Only Changed Rows ===")
print("(In a single trigger(once=True) run, ALL rows are new so both modes look the same.)")
print("(The difference shows up when triggers run incrementally.)\n")
spark.sql("SELECT type, count FROM mode_update ORDER BY count DESC").show(truncate=False)

### Append Mode: Non-Aggregation Queries

Append mode works for queries that do not aggregate -- such as filtering. Each new event that passes the filter is appended to the output.

In [ ]:
# --- Append Mode: Filter only THEFT events (no aggregation) ---
crime_stream3 = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp")) \
    .filter(col("type") == "THEFT")

query_append = crime_stream3.writeStream \
    .format("memory") \
    .queryName("mode_append") \
    .outputMode("append") \
    .trigger(once=True) \
    .start()

query_append.awaitTermination()

print("=== APPEND Mode: Only THEFT Events (no aggregation) ===")
spark.sql("SELECT id, type, district, hour, arrest, event_time FROM mode_append ORDER BY id").show(truncate=False)

### Comprehension Check

**Q1: Why does Spark throw an error if you try to use append mode with a `groupBy().count()` aggregation?**

> *Append mode means "write each row once and never update it." But a running count changes every micro-batch. Writing count=5, then count=6, then count=7 as separate rows would be incorrect -- you want ONE row per group that gets updated.*

**Q2: When would you choose update mode over complete mode in a production system?**

> *Update mode is more efficient when you have many groups but only a few change per batch. Sending the entire table (complete mode) wastes bandwidth if only 2 out of 1000 rows changed. Alerting systems use update mode: "only tell me about groups that changed."*

---
## Part 8: Checkpointing (Concept: Fault Tolerance)

### What Happens When Spark Crashes?

In production, streaming jobs run 24/7. Hardware fails. Nodes crash. Networks partition. What happens to the running counts, the window state, the Kafka offsets?

Without checkpointing: **everything is lost**. Spark has to re-read all data from the beginning.

With checkpointing: Spark periodically saves:
1. **Kafka offsets** -- which messages have been processed.
2. **Aggregation state** -- the running counts, window contents, etc.
3. **Query metadata** -- what the query was doing.

When the job restarts, it reads the checkpoint and resumes **exactly where it left off** -- no data loss, no duplicates (exactly-once semantics).

### How to Enable Checkpointing

Just add `.option("checkpointLocation", "/path/to/checkpoint")` to your `writeStream`. Spark handles the rest automatically.

In [ ]:
import tempfile, os

# Create a temporary checkpoint directory
checkpoint_dir = os.path.join(tempfile.gettempdir(), f"spark_checkpoint_{STUDENT_ID}")
print(f"Checkpoint directory: {checkpoint_dir}\n")

# Re-read and parse
crime_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), crime_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("event_time").cast("timestamp"))

# Checkpointed query
checkpointed_query = crime_stream \
    .groupBy("type").count() \
    .writeStream \
    .format("memory") \
    .queryName("checkpointed_counts") \
    .outputMode("complete") \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(once=True) \
    .start()

checkpointed_query.awaitTermination()

print("=== Checkpointed Query Results ===")
spark.sql("SELECT type, count FROM checkpointed_counts ORDER BY count DESC").show(truncate=False)

# Show checkpoint directory contents
print("=== Checkpoint Directory Contents ===")
for root, dirs, files in os.walk(checkpoint_dir):
    level = root.replace(checkpoint_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files[:5]:  # Show max 5 files per directory
        print(f"{sub_indent}{file}")
    if len(files) > 5:
        print(f"{sub_indent}... and {len(files) - 5} more files")

### Understanding the Checkpoint Directory

The checkpoint directory contains:

- **`offsets/`** -- which Kafka offsets were processed in each micro-batch.
- **`commits/`** -- confirmation that each micro-batch completed successfully.
- **`state/`** -- the aggregation state (running counts, window data, etc.).
- **`metadata`** -- query ID and run metadata.

If Spark crashes and restarts, it reads the latest committed offset, reloads the state, and resumes from exactly that point. No messages are re-processed, no messages are lost.

### Comprehension Check

**Q1: What happens when Spark crashes after processing offset 100 and the last checkpoint is at offset 95? Are messages 96--100 lost or duplicated?**

> *They are reprocessed (offsets 96--100 are read again from Kafka). With checkpointing + write-ahead log, Spark achieves exactly-once semantics: it writes the output and commits the checkpoint atomically. If the commit did not happen, the output is rolled back and reprocessed.*

**Q2: Can two different streaming queries share the same checkpoint directory? Why or why not?**

> *No. Each query must have its own checkpoint directory. The checkpoint stores query-specific state (offsets, aggregations). Sharing would corrupt the state of both queries.*

---
## Part 9: Practice Challenges

Three independent exercises to reinforce what you have learned. Each is wrapped in `try/except` so they do not block the rest of the notebook.

### Challenge 1: Sliding Window

Create a **sliding window** aggregation: 10-minute windows that slide every 2 minutes, grouped by crime type.

A sliding window means events can belong to **multiple** windows. For example, an event at 14:05 belongs to windows starting at 14:00, 13:58, 13:56, etc.

Use `window(col("event_time"), "10 minutes", "2 minutes")` -- the second argument is the window size, the third is the slide interval.

In [ ]:
# CHALLENGE 1: Sliding window -- 10-minute window, 2-minute slide
# Group by (window, type) and count
#
# Hints:
#   - Re-read from Kafka, parse the JSON (copy the pattern from Part 5)
#   - Use window(col("event_time"), "10 minutes", "2 minutes")
#   - Use outputMode("complete") with trigger(once=True)
#   - Write to format("memory"), queryName("sliding_window")
#   - Query with spark.sql("SELECT ... FROM sliding_window ORDER BY window_start")
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Challenge 1 skipped: {e}")

### Challenge 2: Streaming Arrest Rate per District

Compute the **arrest rate** (arrests / total crimes * 100) per district using streaming aggregation.

You need:
- `groupBy("district")`
- `agg(count("*"), sum(col("arrest").cast("int")))`
- Then query the memory table to compute the percentage.

In [ ]:
# CHALLENGE 2: Arrest rate per district
# Steps:
#   1. Re-read and parse the stream
#   2. groupBy("district").agg(count, sum of arrest)
#   3. Write to memory with complete mode
#   4. Query with spark.sql to compute arrest_rate = arrests * 100.0 / total
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Challenge 2 skipped: {e}")

### Challenge 3: Write Results to Parquet

Instead of writing to the console or memory, write the **filtered THEFT events** (non-aggregation) to a Parquet file. Use append mode and a checkpoint location.

This simulates a production pattern: streaming data is written to a data lake in Parquet format for later batch analysis.

In [ ]:
# CHALLENGE 3: Write filtered events to Parquet
# Steps:
#   1. Re-read and parse the stream
#   2. Filter for type == "THEFT"
#   3. writeStream.format("parquet")
#      .option("path", "/tmp/streaming_theft_events")
#      .option("checkpointLocation", "/tmp/theft_checkpoint")
#      .outputMode("append")
#      .trigger(once=True)
#      .start()
#   4. After query completes, read the Parquet back with spark.read.parquet()
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Challenge 3 skipped: {e}")

---
## Part 10: Assessment Preparation

After completing this lab, you should be able to answer the following questions confidently. These cover the core Spark Structured Streaming concepts tested in SE446.

---

**1. What is the difference between `spark.read` and `spark.readStream`?**

> `spark.read` loads a bounded dataset once (batch). `spark.readStream` creates an unbounded streaming DataFrame that continuously ingests new data as it arrives. The API for transformations (filter, groupBy, join) is identical -- only the source and sink change.

**2. Why are the key and value columns in a Kafka DataFrame binary (bytes) instead of strings?**

> Kafka is format-agnostic. It stores raw bytes and does not interpret the content. The producer and consumer agree on the serialization format (JSON, Avro, Protobuf). Spark must cast the bytes to string and then parse the content explicitly.

**3. Why must you define the schema manually with `StructType` for a streaming source?**

> Spark cannot infer the schema of a streaming source because data arrives incrementally -- there is no finite dataset to scan. The schema must be known before the first message arrives so Spark can plan the query.

**4. Explain the difference between event time and processing time. When would each give different results?**

> Event time is when the event occurred (embedded in the data). Processing time is when Spark processes it. They differ when events arrive late (due to network delays, buffering, etc.). A crime at 14:05 processed at 14:30 should be counted in the 14:05 window (event time), not the 14:30 window (processing time).

**5. What is a tumbling window? How is it different from a sliding window?**

> A tumbling window divides time into fixed-size, non-overlapping intervals (e.g., every 5 minutes). Each event belongs to exactly one window. A sliding window has a slide interval smaller than the window size, creating overlapping windows -- an event can belong to multiple windows.

**6. What problem does a watermark solve? What is the trade-off?**

> Without a watermark, Spark must keep state for every window forever (unbounded memory). A watermark declares the maximum expected lateness. Events older than the watermark are dropped, and state for old windows is discarded. Trade-off: smaller watermark = less memory but more dropped late events; larger watermark = more late events accepted but more memory used.

**7. Compare the three output modes: complete, update, and append. When would you use each?**

> Complete: outputs the full result table every trigger -- use for dashboards. Update: outputs only changed rows -- use for alerts. Append: outputs new rows once, never updates -- use for writing to files/data lakes. Aggregation queries cannot use append (counts change). Non-aggregation queries (filters) can use append.

**8. Why does `trigger(once=True)` exist? Why do we use it in this lab?**

> `trigger(once=True)` processes all available data in a single micro-batch and then stops. It is useful for notebooks (continuous streaming would block the cell forever) and for production "catch-up" scenarios where you want streaming semantics on accumulated data.

**9. What does a checkpoint store? What happens if you delete the checkpoint directory?**

> A checkpoint stores Kafka offsets, aggregation state, and query metadata. If deleted, Spark loses track of what was already processed. On restart, it starts from scratch (based on `startingOffsets`), potentially reprocessing data and losing accumulated state.

**10. Your streaming job reads from a Kafka topic with 6 partitions and computes a windowed count. After 30 days, the job is using too much memory. What two things can you do to fix this?**

> (1) Add a watermark to bound the state -- Spark will discard windows older than the watermark threshold. (2) Reduce the number of distinct grouping keys if possible (fewer groups = less state). Additionally, you could increase the window size (fewer windows to track) or use update mode instead of complete mode to avoid re-emitting the entire table.

---

**End of Lab.** You have now connected Spark Structured Streaming to Kafka, parsed JSON messages, performed stateful aggregations, used time windows and watermarks, compared output modes, and set up checkpointing for fault tolerance. These are the core building blocks for any production streaming pipeline.

---
## Cleanup

Run this cell to stop the SparkSession and optionally delete the Kafka topic.

In [ ]:
# --- Stop SparkSession ---
spark.stop()
print("SparkSession stopped.")

# --- Optionally delete the Kafka topic ---
# Uncomment the lines below to clean up

# admin = AdminClient({"bootstrap.servers": BROKER})
# topics_to_delete = [TOPIC]
# futures = admin.delete_topics(topics_to_delete)
# for topic_name, future in futures.items():
#     try:
#         future.result()
#         print(f"Deleted: {topic_name}")
#     except Exception as e:
#         print(f"Could not delete {topic_name}: {e}")

# --- Clean up checkpoint directory ---
# import shutil
# if os.path.exists(checkpoint_dir):
#     shutil.rmtree(checkpoint_dir)
#     print(f"Deleted checkpoint: {checkpoint_dir}")

print("\nDone! You may close this notebook.")